# Eksperimen Machine Learning - Heart Disease Classification
**Nama Peserta:** Dewangga Megananda  
**Dataset:** Heart Disease (UCI Machine Learning Repository)

Notebook ini berisi eksplorasi data (EDA), preprocessing, dan persiapan data untuk modelling.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

# Set style untuk visualisasi
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries berhasil diimport!")

## 2. Load Dataset

In [ ]:
# Load dataset yang sudah dipersiapkan
df = pd.read_csv('dataset_preprocessing/heart_disease_full.csv')

print("Dataset berhasil dimuat!")
print(f"Dimensi dataset: {df.shape}")
print(f"\nInformasi dataset:")
df.info()

In [ ]:
# Tampilkan 5 baris pertama
df.head()

## 3. Exploratory Data Analysis (EDA)

### 3.1 Statistik Deskriptif

In [ ]:
# Statistik deskriptif
print("Statistik Deskriptif:")
df.describe()

### 3.2 Distribusi Target Variable

In [ ]:
# Visualisasi distribusi target
plt.figure(figsize=(8, 6))
ax = sns.countplot(data=df, x='target', palette='Set2')
plt.title('Distribusi Target Variable (Heart Disease)', fontsize=14, fontweight='bold')
plt.xlabel('Target (0 = No Disease, 1 = Disease)', fontsize=12)
plt.ylabel('Count', fontsize=12)

# Tambahkan label pada bar
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.show()

# Persentase
target_counts = df['target'].value_counts()
print(f"\nDistribusi Target:")
print(f"No Disease (0): {target_counts[0]} ({target_counts[0]/len(df)*100:.1f}%)")
print(f"Disease (1): {target_counts[1]} ({target_counts[1]/len(df)*100:.1f}%)")

### 3.3 Analisis Missing Values

In [ ]:
# Cek missing values
missing_data = df.isnull().sum()
missing_percentage = (missing_data / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Values': missing_data,
    'Percentage (%)': missing_percentage
})

print("Missing Values Analysis:")
print(missing_df[missing_df['Missing Values'] > 0])

### 3.4 Analisis Korelasi

In [ ]:
# Korelasi matrix
plt.figure(figsize=(12, 10))
correlation_matrix = df.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', square=True)
plt.title('Correlation Matrix - Heart Disease Dataset', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.5 Distribusi Fitur Numerik

In [ ]:
# Fitur numerik
numeric_features = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

plt.figure(figsize=(15, 10))
for i, feature in enumerate(numeric_features, 1):
    plt.subplot(2, 3, i)
    sns.histplot(data=df, x=feature, hue='target', kde=True, alpha=0.7)
    plt.title(f'Distribution of {feature}', fontsize=12, fontweight='bold')
    plt.xlabel(feature, fontsize=10)
    plt.ylabel('Count', fontsize=10)

plt.tight_layout()
plt.show()

### 3.6 Analisis Fitur Kategorikal

In [ ]:
# Fitur kategorikal
categorical_features = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']

plt.figure(figsize=(15, 12))
for i, feature in enumerate(categorical_features, 1):
    plt.subplot(3, 3, i)
    sns.countplot(data=df, x=feature, hue='target', palette='Set2')
    plt.title(f'{feature} vs Target', fontsize=12, fontweight='bold')
    plt.xlabel(feature, fontsize=10)
    plt.ylabel('Count', fontsize=10)
    plt.legend(title='Target', labels=['No Disease', 'Disease'])

plt.tight_layout()
plt.show()

## 4. Data Preprocessing

### 4.1 Handle Missing Values

In [ ]:
# Cek missing values sebelum preprocessing
print("Missing values sebelum preprocessing:")
print(df.isnull().sum())

# Handle missing values dengan median untuk numerik
numeric_cols = df.select_dtypes(include=[np.number]).columns
imputer = SimpleImputer(strategy='median')
df[numeric_cols] = imputer.fit_transform(df[numeric_cols])

print("\nMissing values setelah preprocessing:")
print(df.isnull().sum())

### 4.2 Encoding Categorical Variables

In [ ]:
# Categorical features yang perlu di-encode
categorical_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']

# Label encoding untuk ordinal features
label_encoder = LabelEncoder()
for col in categorical_cols:
    df[col] = label_encoder.fit_transform(df[col].astype(str))

print("Categorical encoding selesai!")
print(f"\nDimensi dataset setelah encoding: {df.shape}")

### 4.3 Feature Scaling

In [ ]:
# Features yang perlu di-scale (semua kecuali target)
features_to_scale = [col for col in df.columns if col != 'target']

# Standard scaling
scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[features_to_scale] = scaler.fit_transform(df[features_to_scale])

print("Feature scaling selesai!")
print("\nStatistik setelah scaling:")
df_scaled[features_to_scale].describe()

### 4.4 Train/Test Split

In [ ]:
# Split features dan target
X = df_scaled.drop('target', axis=1)
y = df_scaled['target']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train/Test split selesai!")
print(f"Train set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"\nDistribusi target di train set:")
print(y_train.value_counts())
print(f"\nDistribusi target di test set:")
print(y_test.value_counts())

### 4.5 Simpan Preprocessed Data

In [ ]:
# Gabungkan X_train dan y_train untuk disimpan
train_processed = X_train.copy()
train_processed['target'] = y_train

# Gabungkan X_test dan y_test untuk disimpan
test_processed = X_test.copy()
test_processed['target'] = y_test

# Simpan ke CSV
train_processed.to_csv('dataset_preprocessing/train_processed.csv', index=False)
test_processed.to_csv('dataset_preprocessing/test_processed.csv', index=False)

print("Preprocessed data berhasil disimpan!")
print("Files:")
print("- dataset_preprocessing/train_processed.csv")
print("- dataset_preprocessing/test_processed.csv")

## 5. Summary

In [ ]:
print("=== EKSERIMEN PREPROCESSING SUMMARY ===")
print(f"Dataset: Heart Disease Classification")
print(f"Total samples: {len(df)}")
print(f"Features: {len(df.columns)-1}")
print(f"Target classes: {df['target'].nunique()}")
print()
print("Preprocessing steps completed:")
print("✓ Missing value handling")
print("✓ Categorical encoding")
print("✓ Feature scaling")
print("✓ Train/test split")
print("✓ Data export")
print()
print("Files generated:")
print("- train_processed.csv")
print("- test_processed.csv")
print("- Eksperimen_SML_Dewangga_Megananda.txt (https://github.com/nanda910/SMSL_Dewangga_Megananda)")